# 从零训练迷你英中翻译模型 (Seq2Seq)

包含数据生成、模型构建、训练与推理。


In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import random
import itertools

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

## 1. 动态生成 3840 条数据集

In [11]:
def build_dataset():
    animals = {"duck": "鸭子", "cat": "猫", "dog": "狗", "cow": "牛", "bird": "鸟", "horse": "马", "bear": "熊", "lion": "狮子"}
    colors = {"quiet": "安静", "red": "红色", "black": "黑色", "white": "白色", "blue": "蓝色", "green": "绿色", "yellow": "黄色", "brown": "棕色"}
    locations = {"next to": "旁边", "behind": "后面", "in front of": "前面", "on": "上面", "under": "下面", "near": "附近"}
    objects = {"door": "门", "tree": "树", "table": "桌子", "chair": "椅子", "box": "盒子", "window": "窗户", "car": "汽车", "apple": "苹果"}
    names = {"alice": "爱丽丝", "bob": "鲍勃", "charlie": "查理", "david": "大卫", "emma": "艾玛", "fiona": "菲奥娜", "george": "乔治", "henry": "亨利"}
    numbers = {1: ("a", "one", "一"), 2: ("two", "two", "两"), 3: ("three", "three", "三"), 4: ("four", "four", "四"), 5: ("five", "five", "五"), 6: ("six", "six", "六")}

    pattern1_all = []
    for num, color, animal, loc, obj in itertools.product(numbers.keys(), colors.keys(), animals.keys(), locations.keys(), objects.keys()):
        en_num = numbers[num][0]
        zh_num = numbers[num][2]
        if num == 1:
            en = f"there is {en_num} {color} {animal} {loc} the {obj}"
        else:
            en = f"there are {en_num} {color} {animal}s {loc} the {obj}"
        zh = f"{objects[obj]} {locations[loc]} 有 {zh_num} 只 {colors[color]} {animals[animal]}"
        pattern1_all.append((en, zh))

    pattern2_all = []
    for name, num, color, obj in itertools.product(names.keys(), numbers.keys(), colors.keys(), objects.keys()):
        en_num = numbers[num][1]
        zh_num = numbers[num][2]
        if num == 1:
            en = f"{name} has {en_num} {color} {obj}"
        else:
            en = f"{name} has {en_num} {color} {obj}s"
        zh = f"{names[name]} 有 {zh_num} 个 {colors[color]} {objects[obj]}"
        pattern2_all.append((en, zh))

    random.seed(42)
    random.shuffle(pattern1_all)
    random.shuffle(pattern2_all)
    selected = pattern1_all[:1920] + pattern2_all[:1920]
    random.shuffle(selected)
    return selected

raw_data = build_dataset()

# 8:1:1 划分
train_size = int(3840 * 0.8)
val_size = int(3840 * 0.1)
train_data = raw_data[:train_size]
val_data = raw_data[train_size:train_size+val_size]
test_data = raw_data[train_size+val_size:]

print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

# 构建词表
en_vocab = {"<PAD>":0, "<BOS>":1, "<EOS>":2, "<UNK>":3}
zh_vocab = {"<PAD>":0, "<BOS>":1, "<EOS>":2, "<UNK>":3}
for en, zh in train_data:
    for word in en.split():
        if word not in en_vocab: en_vocab[word] = len(en_vocab)
    for word in zh.split():
        if word not in zh_vocab: zh_vocab[word] = len(zh_vocab)

en_idx2word = {v: k for k, v in en_vocab.items()}
zh_idx2word = {v: k for k, v in zh_vocab.items()}
print(f"EN Vocab Size: {len(en_vocab)}, ZH Vocab Size: {len(zh_vocab)}")


Train: 3072, Val: 384, Test: 384
EN Vocab Size: 73, ZH Vocab Size: 51


## 2. 定义 Dataset 与 DataLoader

In [12]:
PAD_IDX = 0
BOS_IDX = 1
EOS_IDX = 2
UNK_IDX = 3

class TranslationDataset(Dataset):
    def __init__(self, data_pairs, en_vocab, zh_vocab):
        self.data_pairs = data_pairs
        self.en_vocab = en_vocab
        self.zh_vocab = zh_vocab
        
    def __len__(self):
        return len(self.data_pairs)
    
    def __getitem__(self, idx):
        en, zh = self.data_pairs[idx]
        en_indices = [self.en_vocab.get(w, UNK_IDX) for w in en.split()]
        zh_indices = [self.zh_vocab.get(w, UNK_IDX) for w in zh.split()]
        
        # 1. 英文反转
        eng_reversed = en_indices[::-1]
        # 为了解决 RNN 中 Padding 污染 Hidden State 的问题，固定左填充到长度 10
        pad_len = max(0, 10 - len(eng_reversed))
        eng_padded = [PAD_IDX] * pad_len + eng_reversed
        # 2. 中文加 BOS 和 EOS
        chn_target = [BOS_IDX] + zh_indices + [EOS_IDX]
        
        return torch.tensor(eng_padded, dtype=torch.long), torch.tensor(chn_target, dtype=torch.long)

def collate_fn(batch):
    eng_batch, chn_batch = zip(*batch)
    eng_padded = torch.nn.utils.rnn.pad_sequence(eng_batch, batch_first=True, padding_value=PAD_IDX)
    chn_padded = torch.nn.utils.rnn.pad_sequence(chn_batch, batch_first=True, padding_value=PAD_IDX)
    return eng_padded, chn_padded

train_dataset = TranslationDataset(train_data, en_vocab, zh_vocab)
val_dataset = TranslationDataset(val_data, en_vocab, zh_vocab)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, collate_fn=collate_fn)


## 3. 模型定义

In [13]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_size=64, hidden_size=128):
        super(Encoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.gru = nn.GRU(embed_size, hidden_size, batch_first=True)

    def forward(self, x):
        embedded = self.embedding(x)
        output, hidden = self.gru(embedded)
        return output, hidden

class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_size=64, hidden_size=128):
        super(Decoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.gru = nn.GRU(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden):
        embedded = self.embedding(x)
        output, hidden = self.gru(embedded, hidden)
        prediction = self.fc(output.squeeze(1))
        return prediction, hidden

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, source, target, teacher_forcing_ratio=0.5):
        batch_size = source.shape[0]
        target_len = target.shape[1]
        target_vocab_size = self.decoder.fc.out_features
        outputs = torch.zeros(batch_size, target_len, target_vocab_size).to(self.device)
        
        _, hidden = self.encoder(source)
        x = target[:, 0].unsqueeze(1)
        
        for t in range(1, target_len):
            prediction, hidden = self.decoder(x, hidden)
            outputs[:, t, :] = prediction
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = prediction.argmax(1)
            x = target[:, t].unsqueeze(1) if teacher_force else top1.unsqueeze(1)
            
        return outputs


## 4. 训练与推理函数

In [14]:
ENG_VOCAB_SIZE = len(en_vocab)
CHN_VOCAB_SIZE = len(zh_vocab)
EMBED_SIZE = 4
HIDDEN_SIZE = 128
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = Encoder(ENG_VOCAB_SIZE, EMBED_SIZE, HIDDEN_SIZE)
decoder = Decoder(CHN_VOCAB_SIZE, EMBED_SIZE, HIDDEN_SIZE)
model = Seq2Seq(encoder, decoder, DEVICE).to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train_epoch(model, dataloader, optimizer, criterion, device, clip=1.0):
    model.train()
    epoch_loss = 0
    for source, target in dataloader:
        source = source.to(device)
        target = target.to(device)
        
        optimizer.zero_grad()
        outputs = model(source, target, teacher_forcing_ratio=0.5)
        
        outputs = outputs[:, 1:].contiguous().view(-1, outputs.shape[-1])
        target = target[:, 1:].contiguous().view(-1)
        
        loss = criterion(outputs, target)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(dataloader)

def translate(model, english_indices, device, max_len=20):
    model.eval()
    with torch.no_grad():
        # 推理时同样固定左填充到长度 10
        eng_reversed = english_indices[::-1]
        pad_len = max(0, 10 - len(eng_reversed))
        eng_padded = [PAD_IDX] * pad_len + eng_reversed
        english_tensor = torch.tensor(eng_padded, dtype=torch.long).unsqueeze(0).to(device)
        _, hidden = model.encoder(english_tensor)
        x = torch.tensor([[BOS_IDX]], dtype=torch.long).to(device)
        predicted_words = []
        for _ in range(max_len):
            prediction, hidden = model.decoder(x, hidden)
            top1 = prediction.argmax(1).item()
            if top1 == EOS_IDX:
                break
            predicted_words.append(top1)
            x = torch.tensor([[top1]], dtype=torch.long).to(device)
    return predicted_words

def evaluate(model, dataloader, criterion, device):
    model.eval()
    epoch_loss = 0
    with torch.no_grad():
        for source, target in dataloader:
            source = source.to(device)
            target = target.to(device)
            
            # 验证时不使用 Teacher Forcing
            outputs = model(source, target, teacher_forcing_ratio=0.0)
            outputs = outputs[:, 1:].contiguous().view(-1, outputs.shape[-1])
            target = target[:, 1:].contiguous().view(-1)
            
            loss = criterion(outputs, target)
            epoch_loss += loss.item()
    return epoch_loss / len(dataloader)


## 5. 执行训练

In [15]:
EPOCHS = 100
PATIENCE = 5
best_val_loss = float("inf")
early_stop_counter = 0

print("开始训练...")
for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss = evaluate(model, val_loader, criterion, DEVICE)
    
    print(f"Epoch [{epoch+1}/{EPOCHS}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        early_stop_counter = 0
        torch.save(model.state_dict(), "best_model.pt")
    else:
        early_stop_counter += 1
        if early_stop_counter >= PATIENCE:
            print(f"连续 {PATIENCE} 轮验证集 Loss 未下降，触发 Early Stopping!")
            break

# 恢复最优模型
model.load_state_dict(torch.load("best_model.pt"))
print("已加载验证集最优模型进行测试。")

print("\n======== 翻译效果演示 ========")
for k in range(10):
    sample_en, sample_zh = test_data[k]
    print("原始英文:", sample_en)
    print("目标中文:", sample_zh)

    sample_en_idx = [en_vocab.get(w, UNK_IDX) for w in sample_en.split()]
    pred_idx = translate(model, sample_en_idx, DEVICE)
    pred_zh = " ".join([zh_idx2word.get(i, "<UNK>") for i in pred_idx])
    print("模型预测:", pred_zh)


开始训练...
Epoch [1/100], Train Loss: 3.6869, Val Loss: 3.2885
Epoch [2/100], Train Loss: 2.8746, Val Loss: 2.4949
Epoch [3/100], Train Loss: 2.2360, Val Loss: 1.9787
Epoch [4/100], Train Loss: 1.7995, Val Loss: 1.6236
Epoch [5/100], Train Loss: 1.5028, Val Loss: 1.3890
Epoch [6/100], Train Loss: 1.3174, Val Loss: 1.2689
Epoch [7/100], Train Loss: 1.2357, Val Loss: 1.2224
Epoch [8/100], Train Loss: 1.1936, Val Loss: 1.1848
Epoch [9/100], Train Loss: 1.1584, Val Loss: 1.1485
Epoch [10/100], Train Loss: 1.1274, Val Loss: 1.1223
Epoch [11/100], Train Loss: 1.1009, Val Loss: 1.0953
Epoch [12/100], Train Loss: 1.0770, Val Loss: 1.0716
Epoch [13/100], Train Loss: 1.0538, Val Loss: 1.0535
Epoch [14/100], Train Loss: 1.0332, Val Loss: 1.0286
Epoch [15/100], Train Loss: 1.0144, Val Loss: 1.0143
Epoch [16/100], Train Loss: 0.9965, Val Loss: 0.9938
Epoch [17/100], Train Loss: 0.9807, Val Loss: 0.9819
Epoch [18/100], Train Loss: 0.9675, Val Loss: 0.9657
Epoch [19/100], Train Loss: 0.9546, Val Loss: 0

## 补充知识点：RNN 编码器中的 Padding 污染问题

在之前的实验中，我们遇到了 `Train Loss` 降得极低，但是**测试推理效果极差**的问题。这是自然语言处理（特别是 RNN/GRU 架构）中非常典型的一个“大坑”——**Padding（填充）污染隐状态 (Hidden State) 问题**。

### 为什么会发生污染？
为了支持批量化计算（Batch Processing），我们要把长短不一的英文句子对齐到同一长度。PyTorch 默认的 `pad_sequence` 会在句子的**末尾补齐 `<PAD>`**（称为右填充）。
* 例如真实句子反转后是：`[windows, white, six, has, henry]`（长度 5）。
* 为了和 Batch 里的最长句子（长度 9）对齐，右填充后变成了：`[windows, white, six, has, henry, <PAD>, <PAD>, <PAD>, <PAD>]`。

**编码器（Encoder）**是一个顺藤摸瓜的结构，它是一个词一个词按顺序读取的。
在**训练**时，GRU 读完最后一个真实词 `henry` 之后，还会继续被迫读取后面的 4 个 `<PAD>`。因此，它最终输出给解码器（Decoder）用来生成中文的语境状态（Hidden State），并不是停留在读完 `henry` 那一刻的感悟，而是被后续 4 个毫无意义的 `<PAD>` 彻底**“冲刷和污染”**过后的状态。模型为了降低 Loss，被迫强行记住并用这种被污染的垃圾状态去翻译出中文。

但在**测试推理（Inference）**时，因为我们一次只传一句英文给模型，没有和其他长句子凑在一起，也就**没有任何 `<PAD>`**。此时 GRU 在读完真实的 `henry` 后立刻输出了最纯净的 Hidden State。然而由于解码器在训练时从没见过这种纯净的状态，它完全不知道该怎么办，于是直接输出了胡言乱语（比如“三只黄色的鸭子”）。

### 我们是如何巧妙解决的？
在工业界，最正统的做法是调用 PyTorch 的 `pack_padded_sequence` 函数，但这对于刚接触 Seq2Seq 的初学者来说概念太过复杂。所以我们在代码中采用了一个极其巧妙的经典替代技巧：**定长左填充 (Fixed-Length Left Padding)**。

我们在 `Dataset` 读取数据和单独 `translate` 测试时，通过代码强制把所有的英文句子都在**左侧（即句首）统一补 `<PAD>`，固定拉长到极限长度（比如 10）**。
* 原句被强行改成了：`[<PAD>, <PAD>, <PAD>, <PAD>, <PAD>, windows, white, six, has, henry]`。

这样一来，无论是在批量训练还是单句推理时，GRU 永远是**一开始先读一堆废话 `<PAD>`，而最后读完的一定是精准的真实单词**！所以，交接给 Decoder 的那个最后一帧 Hidden State，永远都是最纯正、反映了完整真实句意的终极状态。从而彻底让训练和推理的行为达成了 100% 的完美统一！

### 继续实践发现
其实左padding和右padding都可以有很好的预测，但一定要保持训练和推理同时padding，相当于是在同样的padding场景下使用model进行预测，规律就已经是考虑过padding的了。

In [16]:
import torch.nn.functional as F

# 获取预训练好的英文 Embedding 矩阵权重
embed_weights = model.encoder.embedding.weight.data # 形状 (73, 64)

def get_sim(w1, w2):
    idx1 = en_vocab[w1]
    idx2 = en_vocab[w2]
    vec1 = embed_weights[idx1].unsqueeze(0)
    vec2 = embed_weights[idx2].unsqueeze(0)
    return F.cosine_similarity(vec1, vec2).item()

# 比较同类词（颜色 vs 颜色）与异类词（颜色 vs 动物/人名）
print("red 和 black 的相似度 (同为颜色):", round(get_sim("red", "black"), 4))
print("red 和 white 的相似度 (同为颜色):", round(get_sim("red", "white"), 4))
print("red 和 duck  的相似度 (颜色 vs 动物):", round(get_sim("red", "duck"), 4))
print("red 和 henry 的相似度 (颜色 vs 人名):", round(get_sim("red", "henry"), 4))


red 和 black 的相似度 (同为颜色): 0.3953
red 和 white 的相似度 (同为颜色): 0.9275
red 和 duck  的相似度 (颜色 vs 动物): -0.6454
red 和 henry 的相似度 (颜色 vs 人名): 0.709
